In [0]:
TABLE_CUSTOMER_SILVER = "customer_360.silver.customers"
TABLE_CUSTOMER_DIM = "customer_360.dim.dim_customer"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS customer_360.dim
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_CUSTOMER_DIM} (
    customer_sk BIGINT NOT NULL,
    customer_id STRING NOT NULL,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    region STRING,
    customer_segment STRING,
    registration_date DATE,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA
""")

In [0]:
customer_silver=(
    spark
    .read
    .format("delta")
    .table(TABLE_CUSTOMER_SILVER)
)
customer_dim=(
    spark
    .read
    .format("delta")
    .table(TABLE_CUSTOMER_DIM)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *


new_customers=customer_silver.alias("cs").join(
    customer_dim.alias("cd"),
    (col("cs.customer_id") == col("cd.customer_id")) &
    (col("cs.updated_at") == col("cd.effective_from")),
     how="anti"
)

max_sk=customer_dim.agg(max("customer_sk").alias("max_sk")).collect()[0]["max_sk"]

if max_sk is None:
    max_sk = 0

window = Window.orderBy("customer_id")
# display(new_customers)
new_customers = (new_customers.withColumn(
        "customer_sk",
        (row_number().over(Window.orderBy("customer_id")) + lit(max_sk)).cast("long")
    )
    .withColumn(
        "effective_from",
        col("updated_at")
    )
    .withColumn(
        "effective_to",
        lit("9999-12-31 23:59:59").cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
).select(["customer_sk", "customer_id", "first_name", "last_name", "email", "phone", "city", "region", "customer_segment", "registration_date", "effective_from", "effective_to", "is_current"])

In [0]:
new_customers.write.mode("append").saveAsTable(TABLE_CUSTOMER_DIM)

In [0]:

display(
    spark.sql(
        f"SELECT * FROM {TABLE_CUSTOMER_DIM}"
    )
)